In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import flax.nnx as nnx
import jax
import sys
sys.path.append("..")
from NES_VMC_V1 import (
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
)
import jax
import jax.numpy as jnp
from NES_VMC_tool import make_MS_estimator_fn,compute_lam_v_from_samples,level_to_active_col
from H2_6_31G_8Qbit_K4 import SINGLE_SIZE, ha, hi, K, Hatree_Fock, E_fcis, hi_ext

∣NK⟩ Tip: On GPUs an efficient choice is to run ~512 Markov Chains per GPU.

H2 分子基本信息
HF energy = -0.99749729 Ha
Total electrons = (1, 1)
Total basis functions = 4
H₂ FCI 基准能量
E0 = -1.05434745 Ha  |  激发能: 0.0000 eV
E1 = -0.95790573 Ha  |  激发能: 2.6243 eV
E2 = -0.66895227 Ha  |  激发能: 10.4871 eV
E3 = -0.55140192 Ha  |  激发能: 13.6859 eV
E4 = -0.02803278 Ha  |  激发能: 27.9275 eV
E5 = -0.01450343 Ha  |  激发能: 28.2956 eV
E6 = -0.00518297 Ha  |  激发能: 28.5492 eV
E7 = 0.02397075 Ha  |  激发能: 29.3425 eV
E8 = 0.23432390 Ha  |  激发能: 35.0666 eV
E9 = 0.25720575 Ha  |  激发能: 35.6892 eV
E10 = 0.49958128 Ha  |  激发能: 42.2846 eV
E11 = 0.50024484 Ha  |  激发能: 42.3026 eV

HF reference state: [0 0 0 1 0 0 0 1]
single_edges: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]


In [2]:
with open('../data/26-10-07-19-57_history_natural_gradient_H2_molecule_K4__realFFNN.pkl', 'rb') as f:
    history = pickle.load(f)

In [3]:
history.keys()

dict_keys(['steps', 'logpsi_mean', 'logpsi_min', 'logpsi_max', 'grad_norm_raw', 'grad_norm_natural', 'grad_norm_per_param', 'E_L_real', 'E_L_imag', 'loss', 'first_step', 'save_interval', 'Energy_levels', 'params', 'samples'])

## 获取/构建旧态

In [4]:
# ===================== Ansatz：双实输出拼装复数（实参数 + 复输出，替代振幅相位分离版） =====================
class SingleStateAnsatz_SharedBackbone(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_so = n_spin_orbitals
        # 全部网络参数依旧 float 实数，不使用复数权重
        self.backbone1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=float)
        self.backbone2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=float)
        # ========== 修改点：替换原来 head_logA、head_phi → head_u、head_v 两个实输出头 ==========
        self.head_u = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=float)
        self.head_v = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=float)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = nnx.tanh(self.backbone1(x))
        h = nnx.tanh(self.backbone2(h))
        # 两路实输出 u, v
        u = jnp.squeeze(self.head_u(h))
        v = jnp.squeeze(self.head_v(h))
        # ========== 修改点：不再 logA + iφ；拼装复数 z = u + iv，返回 log(z) ==========
        # 小epsilon防止u+iv=0造成log奇点
        z = (u + 1e-12) + 1j * (v + 1e-12)
        return jnp.log(z)



class NESTotalAnsatz_stable(nnx.Module):
    """NES 总拟设：K 个「实参数 + 复输出」单态拟设组成的行列式波函数。"""

    def __init__(self, n_spin_orbitals: int, n_states: int = 2, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = n_states
        self.n_spin = n_spin_orbitals

        self.single_ansatz_list = nnx.List()
        key = rngs.params()

        for _ in range(n_states):
            key, sub_key = jax.random.split(key)
            sub_rngs = nnx.Rngs(params=sub_key)
            ansatz = SingleStateAnsatz_SharedBackbone(n_spin_orbitals, hidden_dim, rngs=sub_rngs)
            self.single_ansatz_list.append(ansatz)

    def __call__(self, x: jax.Array):
        def _forward_single(x_single):
            x_single = x_single.reshape(self.K, self.n_spin)

            # 用 complex128 承接复输出，避免 complex128->complex64 的隐式降精度 scatter
            L = jnp.zeros((self.K, self.K), dtype=jnp.complex128)

            for i in range(self.K):
                for j in range(self.K):
                    L = L.at[i, j].set(self.single_ansatz_list[j](x_single[i]))

            shift = jax.lax.stop_gradient(jnp.max(jnp.real(L)))
            L_stable = L - shift
            Psi_stable = jnp.exp(L_stable)

            sign, log_abs_det = jnp.linalg.slogdet(Psi_stable)
            log_Psi_stable = log_abs_det + 1j * jnp.angle(sign)

            return log_Psi_stable, L_stable, shift

        if x.ndim == 2 and x.shape[-1] == self.n_spin:
            return _forward_single(x)

        elif x.ndim == 2 and x.shape[-1] == self.n_spin * self.K:
            x = x.reshape(-1, self.K, self.n_spin)
            return jax.vmap(_forward_single)(x)

        elif x.ndim == 3:
            x = x.reshape(-1, self.K, self.n_spin)
            return jax.vmap(_forward_single)(x)

        elif x.ndim == 1:
            x = x.reshape(self.K, self.n_spin)
            return _forward_single(x)

        else:
            raise ValueError(f"不支持的输入形状: {x.shape}")


In [5]:

SINGLE_SIZE = 8
K=4
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)
single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]

In [6]:
samples = np.asarray(history['samples'][-2])          # (N, K*SINGLE)，整数构型
N = samples.shape[0]
x_batch = samples.reshape(N, K, SINGLE_SIZE) 


estimator = make_MS_estimator_fn(ha, single_machine_list)
params = history['params'][-1]
Mhat, Shat, finite = estimator(params, x_batch)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/jax/_src/ops/scatter.py:104: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=complex128 to dtype=complex64 with jax_numpy_dtype_promotion=standard. In future JAX releases this will result in an error.
  warnings.warn(


In [7]:
lam_real, v, n_valid , order = compute_lam_v_from_samples(ha=ha,
                           single_machine_list=single_machine_list,
                           total_params=params,
                           x_batch=x_batch,
                           estimator=estimator)
lam_real

array([-1.05500142, -0.95796272, -0.66889533, -0.55074715])

In [8]:
E_fcis[:K]

array([-1.05434745, -0.95790573, -0.66895227, -0.55140192])

In [9]:
history['Energy_levels'][-1]

Array([-1.0543466 +5.85923220e-06j, -0.95790552+2.62164285e-07j,
       -0.66895207+2.34603035e-08j, -0.55140198-5.65633841e-06j],      dtype=complex128)